In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_05 — Model Training + MLflow
# MAGIC **LightGBM primary model → 3 progressive experiments → @Champion → scores 50K users**
# MAGIC
# MAGIC Model strategy:
# MAGIC - **LightGBM** = primary model (best accuracy for tabular credit data)
# MAGIC - Three progressive experiments prove the XScore thesis in numbers
# MAGIC - All tracked in MLflow, v3 becomes @Champion
# MAGIC
# MAGIC **Depends on:** NB_04 complete
# MAGIC **Runtime:** ~12 minutes
# MAGIC **Next:** NB_06_hyperopt

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Install dependencies

# COMMAND ----------

%pip install lightgbm mlflow scikit-learn shap --quiet

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Imports and setup

# COMMAND ----------

import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import numpy as np
import pandas as pd
import json
import pickle
import shap
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.model_selection import train_test_split
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")
mlflow.set_registry_uri("databricks-uc")

print("✓ LightGBM:", lgb.__version__)
print("✓ MLflow  :", mlflow.__version__)
print("✓ Setup complete")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Load feature contract and data

# COMMAND ----------

contract = json.loads(dbutils.fs.head(
    "/Volumes/xscore/bronze/kaggle_raw/feature_contract.json"
))
FEATURE_COLS = contract["feature_cols"]
LABEL_COL    = contract["label_col"]

print(f"Feature contract v{contract['version']}: {len(FEATURE_COLS)} features")

# Load as Pandas — 50K rows fits in driver memory easily
gold_pd = (spark.table("xscore.gold.credit_feature_store")
           .select(["user_id", "segment"] + FEATURE_COLS + [LABEL_COL])
           .fillna(0.0)
           .toPandas())

print(f"Loaded {len(gold_pd):,} rows")
print(f"Default rate: {gold_pd[LABEL_COL].mean():.2%}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Train / val / test split

# COMMAND ----------

X = gold_pd[FEATURE_COLS]
y = gold_pd[LABEL_COL]

# Stratified split — keeps same default rate in each split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15/0.85, random_state=42, stratify=y_trainval
)

print(f"Train : {len(X_train):,}  ({y_train.mean():.2%} default)")
print(f"Val   : {len(X_val):,}   ({y_val.mean():.2%} default)")
print(f"Test  : {len(X_test):,}   ({y_test.mean():.2%} default)")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Helper functions and base params

# COMMAND ----------

def train_lgbm(X_tr, y_tr, X_v, y_v, params):
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_v,  label=y_v, reference=dtrain)
    model  = lgb.train(
        params,
        dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(0),
        ],
    )
    return model

def eval_lgbm(model, X, y):
    prob = model.predict(X)
    pred = (prob >= 0.5).astype(int)
    return {
        "auc_roc": round(roc_auc_score(y, prob), 4),
        "auc_pr" : round(average_precision_score(y, prob), 4),
        "f1"     : round(f1_score(y, pred, zero_division=0), 4),
    }

# Class imbalance correction — default is rare (~9%)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

BASE_PARAMS = {
    "objective"        : "binary",
    "metric"           : "auc",
    "boosting_type"    : "gbdt",
    "learning_rate"    : 0.05,
    "num_leaves"       : 31,
    "max_depth"        : 6,
    "min_child_samples": 20,
    "feature_fraction" : 0.80,
    "bagging_fraction" : 0.80,
    "bagging_freq"     : 5,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 0.1,
    "scale_pos_weight" : pos_weight,
    "random_state"     : 42,
    "verbose"          : -1,
}
print(f"✓ scale_pos_weight: {pos_weight:.2f}  (handles class imbalance)")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — v1: Baseline (traditional features only)

# COMMAND ----------

V1_FEATURES = [
    "income_log", "employment_capped",
    "owns_land_d", "owns_vehicle_d",
    "bank_vintage_capped", "itr_filed_d",
    "sim_tenure_months",
]

mlflow.set_experiment("/Users/aryamanbhati8@gmail.com/xscore_credit_scoring_lgbm")

with mlflow.start_run(run_name="v1_baseline") as run:
    mlflow.log_params({
        "model": "lightgbm", "feature_set": "baseline",
        "n_features": len(V1_FEATURES),
        "description": "Income + employment + assets only. No novel data.",
    })
    lgbm_v1 = train_lgbm(
        X_train[V1_FEATURES], y_train,
        X_val[V1_FEATURES],   y_val, BASE_PARAMS
    )
    val_v1  = eval_lgbm(lgbm_v1, X_val[V1_FEATURES],  y_val)
    test_v1 = eval_lgbm(lgbm_v1, X_test[V1_FEATURES], y_test)
    mlflow.log_metrics({**{f"val_{k}": v for k, v in val_v1.items()},
                        **{f"test_{k}": v for k, v in test_v1.items()},
                        "best_iteration": lgbm_v1.best_iteration})
    
    signature = infer_signature(X_train[V1_FEATURES], lgbm_v1.predict(X_train[V1_FEATURES]))
    mlflow.lightgbm.log_model(
        lgbm_v1, "model",
        signature=signature,
        input_example=X_train[V1_FEATURES].head(5)
    )
    RUN_ID_V1 = run.info.run_id

print(f"v1  AUC: {val_v1['auc_roc']:.4f}  F1: {val_v1['f1']:.4f}  iters: {lgbm_v1.best_iteration}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — v2: Add bill payment (Pillar 1)

# COMMAND ----------

V2_FEATURES = V1_FEATURES + [
    "p1_bill_ontime_rate", "p1_avg_days_late",
    "p1_severe_late_rate", "p1_bill_type_diversity",
    "p1_payment_trend",
]

with mlflow.start_run(run_name="v2_add_bills") as run:
    mlflow.log_params({
        "model": "lightgbm", "feature_set": "baseline_plus_bills",
        "n_features": len(V2_FEATURES),
        "added": "5 bill payment features (Pillar 1)",
    })
    lgbm_v2 = train_lgbm(
        X_train[V2_FEATURES], y_train,
        X_val[V2_FEATURES],   y_val, BASE_PARAMS
    )
    val_v2  = eval_lgbm(lgbm_v2, X_val[V2_FEATURES],  y_val)
    test_v2 = eval_lgbm(lgbm_v2, X_test[V2_FEATURES], y_test)
    lift_v2 = val_v2["auc_roc"] - val_v1["auc_roc"]
    mlflow.log_metrics({**{f"val_{k}": v for k, v in val_v2.items()},
                        **{f"test_{k}": v for k, v in test_v2.items()},
                        "auc_lift_vs_v1": lift_v2,
                        "best_iteration": lgbm_v2.best_iteration})
    
    signature = infer_signature(X_train[V2_FEATURES], lgbm_v2.predict(X_train[V2_FEATURES]))
    mlflow.lightgbm.log_model(
        lgbm_v2, "model",
        signature=signature,
        input_example=X_train[V2_FEATURES].head(5)
    )
    RUN_ID_V2 = run.info.run_id

print(f"v2  AUC: {val_v2['auc_roc']:.4f}  F1: {val_v2['f1']:.4f}  lift: +{lift_v2:.4f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — v3: Full XScore — all 6 pillars (Champion candidate)

# COMMAND ----------

V3_PARAMS = {**BASE_PARAMS,
             "learning_rate": 0.04, "num_leaves": 63,
             "max_depth": 7, "min_child_samples": 15,
             "feature_fraction": 0.75}

with mlflow.start_run(run_name="v3_full_xscore") as run:
    mlflow.log_params({
        "model": "lightgbm", "feature_set": "full_xscore_6_pillars",
        "n_features": len(FEATURE_COLS),
        "pillars": "bills, upi, assets, income, identity, stability + composites",
        **{f"lgbm_{k}": v for k, v in V3_PARAMS.items() if k != "verbose"},
    })
    lgbm_v3 = train_lgbm(
        X_train[FEATURE_COLS], y_train,
        X_val[FEATURE_COLS],   y_val, V3_PARAMS
    )
    val_v3  = eval_lgbm(lgbm_v3, X_val[FEATURE_COLS],  y_val)
    test_v3 = eval_lgbm(lgbm_v3, X_test[FEATURE_COLS], y_test)
    lift_v3_v1 = val_v3["auc_roc"] - val_v1["auc_roc"]
    lift_v3_v2 = val_v3["auc_roc"] - val_v2["auc_roc"]

    # Feature importance
    fi = pd.DataFrame({
        "feature"   : FEATURE_COLS,
        "importance": lgbm_v3.feature_importance(importance_type="gain"),
    }).sort_values("importance", ascending=False)
    fi.to_csv("/tmp/feature_importances.csv", index=False)

    mlflow.log_metrics({**{f"val_{k}": v for k, v in val_v3.items()},
                        **{f"test_{k}": v for k, v in test_v3.items()},
                        "auc_lift_vs_v1": lift_v3_v1,
                        "auc_lift_vs_v2": lift_v3_v2,
                        "best_iteration": lgbm_v3.best_iteration})
    mlflow.log_artifact("/tmp/feature_importances.csv")
    
    signature = infer_signature(X_train[FEATURE_COLS], lgbm_v3.predict(X_train[FEATURE_COLS]))
    mlflow.lightgbm.log_model(
        lgbm_v3, "model",
        signature=signature,
        input_example=X_train[FEATURE_COLS].head(5),
        registered_model_name="xscore.gold.credit_scorer"
    )
    RUN_ID_V3 = run.info.run_id

print(f"v3  AUC: {val_v3['auc_roc']:.4f}  F1: {val_v3['f1']:.4f}  lift: +{lift_v3_v1:.4f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — The thesis table (show judges this)

# COMMAND ----------

print()
print("=" * 75)
print("  XSCORE — ALTERNATIVE DATA IMPROVES CREDIT SCORING")
print("=" * 75)
print(f"  {'Version':<20} {'Feats':>5} {'Val AUC':>9} {'Val F1':>8} {'Lift':>8}  What was added")
print("  " + "─" * 70)
print(f"  {'v1 Baseline':<20} {len(V1_FEATURES):>5} {val_v1['auc_roc']:>9.4f} {val_v1['f1']:>8.4f} {'—':>8}  Income + employment only")
print(f"  {'v2 + Bill payment':<20} {len(V2_FEATURES):>5} {val_v2['auc_roc']:>9.4f} {val_v2['f1']:>8.4f} {lift_v2:>+8.4f}  Utility/mobile bill history")
print(f"  {'v3 Full XScore':<20} {len(FEATURE_COLS):>5} {val_v3['auc_roc']:>9.4f} {val_v3['f1']:>8.4f} {lift_v3_v1:>+8.4f}  UPI + govt + stability + composites")
print("  " + "─" * 70)
print(f"  Total AUC lift from XScore alternative data: +{lift_v3_v1:.4f}")
print()
print("  Bills, UPI transactions, govt scheme participation —")
print("  none of this data exists in CIBIL. That is the XScore value prop.")
print("=" * 75)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 10 — SHAP values and top features

# COMMAND ----------

print("Computing SHAP values on validation sample...")

sample_X = X_val[FEATURE_COLS].sample(n=min(1000, len(X_val)), random_state=42)
explainer   = shap.TreeExplainer(lgbm_v3)
shap_values = explainer.shap_values(sample_X)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_imp = pd.DataFrame({
    "feature"      : FEATURE_COLS,
    "mean_abs_shap": np.abs(sv).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

shap_imp["pillar"] = shap_imp["feature"].apply(lambda f:
    "P1-Bills"    if f.startswith("p1_") else
    "P2-UPI"      if f.startswith("p2_") else
    "P3-Assets"   if f in ["owns_land_d","land_acres_capped","owns_vehicle_d","bank_vintage_capped","has_fd_or_rd_d"] else
    "P4-Income"   if f in ["income_log","itr_filed_d","gst_registered_d","employment_capped"] else
    "P5-Identity" if f in ["jan_dhan_active_d","shg_member_d","shg_months","dbt_months","svanidhi_repaid_d"] else
    "P6-Stability"if f in ["sim_tenure_months","location_stability","fraud_flag_d"] else
    "Derived"
)

print("\nTop 15 features by SHAP importance:")
print(shap_imp[["feature","pillar","mean_abs_shap"]].head(15).to_string(index=False))

# Save explainer for NB_08
with open("/Volumes/xscore/bronze/kaggle_raw/lgbm_v3_explainer.pkl", "wb") as f:
    pickle.dump(explainer, f)

shap_imp.to_csv("/tmp/shap_importance.csv", index=False)
with mlflow.start_run(run_id=RUN_ID_V3):
    mlflow.log_artifact("/tmp/shap_importance.csv")

print("\n✓ SHAP explainer saved to Volume for NB_08")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 11 — Register @Champion

# COMMAND ----------

client  = MlflowClient()
version = max(int(v.version) for v in
              client.search_model_versions("name='xscore.gold.credit_scorer'"))

client.set_registered_model_alias("xscore.gold.credit_scorer", "Champion", str(version))
client.set_model_version_tag("xscore.gold.credit_scorer", str(version), "val_auc_roc",       f"{val_v3['auc_roc']:.4f}")
client.set_model_version_tag("xscore.gold.credit_scorer", str(version), "model_type",        "lightgbm")
client.set_model_version_tag("xscore.gold.credit_scorer", str(version), "feature_contract",  "v1.0")

print(f"✓ @Champion → version {version}  AUC: {val_v3['auc_roc']:.4f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 12 — Score all 50K users, MERGE into gold.credit_scores

# COMMAND ----------

probs = lgbm_v3.predict(gold_pd[FEATURE_COLS].fillna(0.0))

gold_pd["default_probability"] = probs
gold_pd["xscore"]     = (900 - probs * 900).astype(int)
gold_pd["score_band"] = pd.cut(
    gold_pd["xscore"],
    bins=[-1, 400, 600, 750, 901],
    labels=["Poor", "Fair", "Good", "Excellent"]
).astype(str)
gold_pd["score_timestamp"] = pd.Timestamp.now()
gold_pd["model_version"]   = f"v3_lgbm_v{version}"
gold_pd["model_run_id"]    = RUN_ID_V3

scores_sp = spark.createDataFrame(
    gold_pd[["user_id","segment","xscore","score_band",
             "default_probability","score_timestamp",
             "model_version","model_run_id"]]
)

DeltaTable.forName(spark, "xscore.gold.credit_scores") \
    .alias("t").merge(scores_sp.alias("s"), "t.user_id = s.user_id") \
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

n = spark.table("xscore.gold.credit_scores").count()
print(f"✓ {n:,} users scored and written to gold.credit_scores")

display(spark.table("xscore.gold.credit_scores")
    .groupBy("score_band")
    .agg(F.count("*").alias("users"),
         F.round(F.avg("xscore"),0).alias("avg_xscore"),
         F.round(F.avg("default_probability")*100,2).alias("avg_default_pct"))
    .orderBy("avg_xscore", ascending=False))

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 13 — Delta time travel demo

# COMMAND ----------

# MAGIC %sql
# MAGIC DESCRIBE HISTORY xscore.gold.credit_scores;

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 14 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_05 COMPLETE")
print("=" * 60)
print(f"  Model     : LightGBM v{version} @Champion")
print(f"  Val AUC   : {val_v3['auc_roc']:.4f}")
print(f"  Val F1    : {val_v3['f1']:.4f}")
print(f"  AUC lift  : +{lift_v3_v1:.4f} vs baseline")
print(f"  Scored    : {n:,} users")
print()
print("  NEXT: NB_06_hyperopt")
print("  Hyperopt will search for better params and")
print("  auto-promote @Champion if AUC improves.")
print("=" * 60)